# 26 (MLA) — Classification Workflow

**ML Analyst perspective.** Train a `LogisticRegression` on `inadimplente`, inspect probability vs prediction, and evaluate with accuracy/precision/recall/F1/AUC. Then explore how the decision **threshold** trades precision against recall.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Load splits + train

`LogisticRegression` fits in numpy (gradient descent, L2-regularized).

In [ ]:
train = session.table("mla_train")
test = session.table("mla_test")
from irispark.ml.classification import LogisticRegression

feats = ["renda_std", "idade_std", "historico_idx_std", "divida_ratio_std"]
lr = LogisticRegression(featuresCol=feats, labelCol="inadimplente", maxIter=500, learningRate=0.5, regParam=0.1)
model = lr.fit(train)
print("backend:", model.backend)

## 2. Probability + prediction

`transform` adds a `probability` column (SQL sigmoid) and a `prediction` column (thresholded).

In [ ]:
pred = model.transform(test)
pred.select("cliente_id", "inadimplente", "probability", "prediction").show(5)

## 3. Evaluate

Accuracy, precision, recall, F1, and ROC AUC.

In [ ]:
from irispark.ml.evaluation import BinaryClassificationEvaluator

for m in ("accuracy", "precision", "recall", "f1", "areaUnderROC"):
    ev = BinaryClassificationEvaluator(predictionCol="prediction", labelCol="inadimplente", metricName=m)
    print(f"{m}: {ev.evaluate(pred):.3f}")

## 4. Threshold sensitivity

Lowering the threshold catches more defaults (higher recall) but flags more false positives (lower precision).

In [ ]:
for th in (0.3, 0.5, 0.7):
    m = LogisticRegression(featuresCol=feats, labelCol="inadimplente", maxIter=500, learningRate=0.5, regParam=0.1, threshold=th).fit(train)
    p = m.transform(test)
    acc = BinaryClassificationEvaluator(predictionCol="prediction", labelCol="inadimplente", metricName="accuracy").evaluate(p)
    rec = BinaryClassificationEvaluator(predictionCol="prediction", labelCol="inadimplente", metricName="recall").evaluate(p)
    prec = BinaryClassificationEvaluator(predictionCol="prediction", labelCol="inadimplente", metricName="precision").evaluate(p)
    print(f"threshold={th}: accuracy={acc:.3f} precision={prec:.3f} recall={rec:.3f}")

## 5. Class balance

If defaults are rare, accuracy can be misleading — precision/recall/AUC matter more. Check the balance.

In [ ]:
train.groupBy("inadimplente").count().orderBy("inadimplente").show()

## 6. What's next

The same loop extends to tree ensembles (Phase 8) and AutoML for automatic model selection.

In [ ]:
print("classification workflow complete")

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")